In [6]:
# =============================================================================
# CELL 1: IMPORTS AND CONFIGURATION
# =============================================================================

import pandas as pd
import numpy as np
import pyodbc
import warnings

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', '{:.0f}'.format)

def run_sql(query):
    with pyodbc.connect("DSN=Redshift_prod_new") as conn:
        warnings.filterwarnings("ignore", category=UserWarning)
        df = pd.read_sql_query(sql=query, con=conn)
        warnings.filterwarnings("default", category=UserWarning)
    return df

print('Imports ready')

Imports ready


In [7]:
# =============================================================================
# CELL 2: DEFINE CORPORATE DEALER GROUPS
# =============================================================================

def expand_dealer_spec(spec_str):
    """Parse a dealer specification string into a list of dealer numbers.
    Commas separate entries. Dashes/en-dashes indicate inclusive ranges.
    """
    dealers = []
    parts = [p.strip() for p in spec_str.split(',')]
    for part in parts:
        part = part.replace('\u2013', '-').replace('\u2014', '-')
        if '-' in part:
            tokens = part.split('-')
            start, end = int(tokens[0].strip()), int(tokens[-1].strip())
            dealers.extend(range(start, end + 1))
        else:
            dealers.append(int(part))
    return dealers


DEALER_GROUPS = {
    'Auto Boutique': '29951, 30405, 31477',
    'Avis': '28558, 28563, 28812-28822, 28827-28829, 29005-29011, 29533, 30282, 33347',
    'EchoPark': '29196-29198, 31182, 31187, 31193, 31196, 31205, 31708, 31776, 33106',
    'HGreg': '28347-28351, 29646-29647, 29829, 29842-29843, 30211, 33325',
    'Hertz Car Sales': '27412-27419, 27424-27426, 28234, 28411-28536, 28883, 29013-29022, 29452, 29609, 30663-30669, 31169, 32845, 58934-58935, 60103, 60107',
    'Penske': '6333, 16099, 16102, 16103, 23328, 23329, 23330, 23331, 23332, 23334, 23335, 23336, 23337, 23339, 23340, 23341, 23342, 23343, 23383, 23397, 23398, 23399, 23400, 23401, 23402, 23403, 23418, 23430, 23431, 23432, 23433, 23434, 23435, 23436, 23459, 23464, 23465, 23466, 23467, 23468, 23469, 23471, 23472, 23473, 23500, 23523, 23669, 23670, 23671, 23672, 23674, 23675, 23677, 23707, 23708, 23709, 23710, 23711, 23712, 23713, 23726, 23728, 23729, 23730, 23731, 23732, 23733, 23734, 23735, 23736, 23737, 23738, 23757, 23758, 23759, 23760, 23762, 23763, 23765, 23766, 23767, 23768, 23769, 23770, 23771, 23772, 23773, 23774, 23775, 23776, 23777, 23815, 23816, 23817, 23818, 23819, 23820, 23821, 23822, 23823, 23824, 23825, 23826, 23827, 23828, 23829, 23830, 23831, 23832, 23833, 23834, 23835, 24323, 24562, 24563, 24564, 25203, 26075, 26204, 26205, 26206, 26207, 26208, 26291, 26346, 26639, 26781, 27148, 27483, 27484, 28713, 29786, 29955, 30198, 30281, 30639, 30944, 30945, 31088, 31089, 31287, 31288, 31316, 31742, 32396, 32893, 32894, 33386, 33431, 33432, 39055, 39273, 45038, 53024, 55250, 55482, 63098',
    'Woodhouse Auto Family': '28919, 28959-28967, 29282-29284, 29937, 31007, 32131, 32466, 32866, 40301, 45656, 46274-46275, 47115, 53065-53066',
}

SONIC_DEALERS = [
    18, 61, 66, 91, 96, 102, 136, 226, 262, 341, 400, 401, 405, 406, 408,
    411, 412, 413, 414, 415, 416, 426, 432, 433, 435, 436, 437, 442, 443,
    444, 445, 446, 447, 450, 454, 455, 456, 457, 458, 459, 460, 467, 468,
    469, 475, 478, 480, 482, 483, 484, 485, 486, 487, 489, 490, 491, 492,
    493, 494, 495, 496, 497, 498, 499, 500, 501, 502, 503, 504, 505, 506,
    507, 508, 509, 510, 511, 512, 513, 515, 516, 517, 518, 523, 524, 527,
    528, 529, 530, 531, 532, 533, 535, 537, 538, 539, 541, 542, 543, 560,
    635, 636, 638, 639, 640, 719, 734, 758, 789, 799, 938, 942, 1080, 1091,
    1121, 8395, 29127, 29128, 29129, 29130, 29131, 29132, 29133, 29134,
    29135, 29136, 29137, 29139, 29140, 29141, 29142, 29143, 29144, 29145,
    29147, 29148, 29149, 29150, 29151, 29152, 29153, 29154, 29155, 29156,
    29157, 29158, 29159, 29160, 29161, 29162, 29165, 29166, 29167, 29168,
    29169, 29170, 29171, 29172, 29173, 29174, 29175, 29176, 29177, 29178,
    29179, 29180, 29181, 29182, 29183, 29184, 29185, 29186, 29187, 29188,
    29189, 29190, 29191, 29192, 29193, 29194, 29195, 29196, 29197, 29198,
    29199, 29200, 29202, 29203, 29204, 29205, 29206, 29207, 29208, 29209,
    29210, 29211, 29212, 29213, 29215, 29216, 29217, 29218, 29483, 30266,
    30469, 30582, 30593, 30600, 30746, 30961, 30962, 30963, 30964, 30965,
    30966, 30967, 30968, 30969, 30970, 30971, 30972, 30973, 30974, 30975,
    30976, 30977, 30978, 30979, 30980, 30981, 30982, 30983, 30984, 30985,
    31011, 31182, 31183, 31184, 31185, 31186, 31187, 31188, 31189, 31190,
    31191, 31192, 31193, 31194, 31195, 31196, 31197, 31198, 31199, 31200,
    31201, 31202, 31203, 31204, 31205, 31206, 31207, 31208, 31209, 31210,
    31211, 31212, 31213, 31214, 31215, 31350, 31776, 32382, 33106, 35549,
    35550, 35551, 35552, 35553, 35554, 35555, 35556, 35557, 37913, 41945,
    49860, 49927, 59479, 59746, 59751, 59752, 59753, 67049, 67051, 67053,
    67054, 67057,
]

expanded_groups = {}
for name, spec in DEALER_GROUPS.items():
    expanded = expand_dealer_spec(spec)
    expanded_groups[name] = expanded
    print(f'{name}: {len(expanded):,} dealer numbers')

expanded_groups['Sonic Automotive'] = SONIC_DEALERS
print(f'Sonic Automotive: {len(SONIC_DEALERS):,} dealer numbers')

# Build reverse lookup: dealer_number -> group name
dealer_to_group = {}
for group_name, dealer_list in expanded_groups.items():
    for d in dealer_list:
        dealer_to_group[d] = group_name

all_dealer_numbers = sorted(dealer_to_group.keys())
print(f'\nTotal dealer groups: {len(expanded_groups)}')
print(f'Total unique dealer numbers across all groups: {len(all_dealer_numbers):,}')

Auto Boutique: 3 dealer numbers
Avis: 26 dealer numbers
EchoPark: 11 dealer numbers
HGreg: 12 dealer numbers
Hertz Car Sales: 164 dealer numbers
Penske: 157 dealer numbers
Woodhouse Auto Family: 25 dealer numbers
Sonic Automotive: 295 dealer numbers

Total dealer groups: 8
Total unique dealer numbers across all groups: 683


In [8]:
# =============================================================================
# CELL 3: QUERY APPLICATION DATA FROM REDSHIFT
# =============================================================================

dealer_list_sql = ', '.join(str(d) for d in all_dealer_numbers)

query = f"""
SELECT cd.dealer_number,
       DATE(cd.application_received_dtm) AS app_date,
       cd.loan_id
FROM edwnpi.los_deal_current_fact AS cd
WHERE cd.application_received_dtm >= '2026-01-01'
  AND cd.data_source_name != 'SPARTAN'
  AND cd.dealer_number IN ({dealer_list_sql})
"""

apps_df = run_sql(query)
apps_df['app_date'] = pd.to_datetime(apps_df['app_date'])
print(f'Application records fetched: {len(apps_df):,}')
print(f'Date range: {apps_df.app_date.min()} to {apps_df.app_date.max()}')
print(f'Unique dealers with applications: {apps_df.dealer_number.nunique():,}')

Application records fetched: 169,878
Date range: 2026-01-01 00:00:00 to 2026-09-11 00:00:00
Unique dealers with applications: 349


In [9]:
# =============================================================================
# CELL 4: MAP DEALERS TO GROUPS AND COMPUTE WEEKLY APP COUNTS
# =============================================================================

apps_df['dealer_group'] = apps_df['dealer_number'].map(dealer_to_group)
apps_df['week'] = apps_df['app_date'].dt.to_period('W-SAT').astype(str)

weekly_apps = (
    apps_df
    .groupby(['dealer_group', 'week'])
    .agg(app_count=('loan_id', 'nunique'))
    .reset_index()
)

pivot = weekly_apps.pivot(index='dealer_group', columns='week', values='app_count').fillna(0).astype(int)
pivot['Total'] = pivot.sum(axis=1)
pivot = pivot.sort_values('Total', ascending=False)

# Add a row with weekly totals across all groups
week_totals = pivot.sum(axis=0)
week_totals.name = 'All Groups'
pivot = pd.concat([pivot, week_totals.to_frame().T])

print(f'Weekly application counts by dealer group (2026):')
print(f'Weeks covered: {pivot.shape[1] - 1}')
display(pivot)

Weekly application counts by dealer group (2026):
Weeks covered: 37


week,2025-12-28/2026-01-03,2026-01-04/2026-01-10,2026-01-11/2026-01-17,2026-01-18/2026-01-24,2026-01-25/2026-01-31,2026-02-01/2026-02-07,2026-02-08/2026-02-14,2026-02-15/2026-02-21,2026-02-22/2026-02-28,2026-03-01/2026-03-07,2026-03-08/2026-03-14,2026-03-15/2026-03-21,2026-03-22/2026-03-28,2026-03-29/2026-04-04,2026-04-05/2026-04-11,2026-04-12/2026-04-18,2026-04-19/2026-04-25,2026-04-26/2026-05-02,2026-05-03/2026-05-09,2026-05-10/2026-05-16,2026-05-17/2026-05-23,2026-05-24/2026-05-30,2026-05-31/2026-06-06,2026-06-07/2026-06-13,2026-06-14/2026-06-20,2026-06-21/2026-06-27,2026-06-28/2026-07-04,2026-07-05/2026-07-11,2026-07-12/2026-07-18,2026-07-19/2026-07-25,2026-07-26/2026-08-01,2026-08-02/2026-08-08,2026-08-09/2026-08-15,2026-08-16/2026-08-22,2026-08-23/2026-08-29,2026-08-30/2026-09-05,2026-09-06/2026-09-12,Total
Sonic Automotive,869,2640,2508,2294,2611,2628,2798,3543,5091,3634,3167,2991,2898,2417,2611,2367,2397,2280,2406,2166,2486,2166,2117,2107,2224,2357,2070,2254,2377,2624,2369,2067,2103,1903,1963,1976,1182,90661
Hertz Car Sales,127,466,400,416,458,541,479,737,1068,1000,1013,858,1033,1020,1052,881,856,879,807,659,831,784,684,639,633,742,725,705,707,621,511,507,592,547,490,538,406,25412
HGreg,165,398,474,502,501,585,558,673,1126,870,751,676,670,579,488,497,521,510,391,494,414,448,361,463,461,461,380,440,438,439,384,411,430,356,293,304,215,18127
Auto Boutique,116,245,338,330,332,397,333,459,691,570,446,358,373,408,325,306,319,295,343,310,311,214,311,229,259,260,235,320,281,313,300,260,303,315,349,297,197,12048
Penske,94,223,262,247,318,268,387,513,591,487,477,415,369,359,313,359,378,347,296,312,292,289,350,271,360,354,261,313,274,316,283,291,323,306,293,290,164,12045
Woodhouse Auto Family,13,58,65,51,56,72,70,75,84,46,64,68,73,56,48,62,56,73,54,65,54,33,59,58,76,49,48,48,53,46,69,53,53,58,50,54,33,2103
Avis,21,72,29,31,50,60,56,72,72,49,47,46,30,31,27,36,32,20,28,41,23,35,47,23,52,37,40,43,53,65,38,40,37,29,39,29,15,1495
All Groups,1405,4102,4076,3871,4326,4551,4681,6072,8723,6656,5965,5412,5446,4870,4864,4508,4559,4404,4325,4047,4411,3969,3929,3790,4065,4260,3759,4123,4183,4424,3954,3629,3841,3514,3477,3488,2212,161891


In [10]:
# =============================================================================
# CELL 5: SUMMARY TABLE (LONG FORMAT)
# =============================================================================

summary = (
    weekly_apps
    .groupby('dealer_group')
    .agg(
        total_apps=('app_count', 'sum'),
        avg_weekly_apps=('app_count', 'mean'),
        min_weekly_apps=('app_count', 'min'),
        max_weekly_apps=('app_count', 'max'),
        weeks_with_apps=('app_count', lambda x: (x > 0).sum()),
    )
    .sort_values('total_apps', ascending=False)
)

summary['avg_weekly_apps'] = summary['avg_weekly_apps'].round(1)
print('Summary by dealer group:')
display(summary)

Summary by dealer group:


,total_apps,avg_weekly_apps,min_weekly_apps,max_weekly_apps,weeks_with_apps
dealer_group,,,,,
Sonic Automotive,90661,2450,869,5091,37
Hertz Car Sales,25412,687,127,1068,37
HGreg,18127,490,165,1126,37
Auto Boutique,12048,326,116,691,37
Penske,12045,326,94,591,37
Woodhouse Auto Family,2103,57,13,84,37
Avis,1495,40,15,72,37
